# Notebook 03: Feature Selection Methods

## Why This Matters

Adding more features does not always help — it can hurt. Irrelevant features add noise, increase training time, and worsen generalization through the curse of dimensionality. High-dimensional feature spaces also slow inference in production and inflate model storage costs. Feature selection is therefore both an accuracy technique and an engineering discipline, and it appears in virtually every applied ML interview: "how would you reduce a 500-feature dataset to the most useful 50?" Understanding the three families of methods — filter, wrapper, and embedded — and their trade-offs is essential for answering that question credibly.

## 1. The Three Families of Feature Selection

Feature selection methods are categorized by how they interact with the model:

| Family | Mechanism | Pros | Cons |
|--------|-----------|------|------|
| **Filter** | Score each feature independently of any model (correlation, mutual info, chi²) | Fast; model-agnostic; no overfitting | Ignores feature interactions; misses redundancy |
| **Wrapper** | Use a model to evaluate subsets (RFE, forward/backward selection) | Captures interactions; model-specific | Expensive (trains model many times); can overfit to training set |
| **Embedded** | Selection happens during model training (L1 / Lasso, tree feature importance) | Fast; captures interactions; built-in regularization | Tied to specific model class |

In practice, a pipeline typically applies a fast filter first (remove near-zero-variance features), then an embedded method (Lasso or tree importance), optionally followed by wrapper validation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Create a dataset: 20 features, only 5 are truly informative
X, y = make_classification(
    n_samples=1000, n_features=20, n_informative=5,
    n_redundant=5, n_repeated=2, n_clusters_per_class=1,
    random_state=42
)
feature_names = [f'f{i:02d}' for i in range(20)]
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y

print("Dataset shape:", X.shape)
print("Class balance:", np.bincount(y))

# Baseline: use all features
pipe_base = LogisticRegression(max_iter=500)
sc = StandardScaler()
scores_all = cross_val_score(pipe_base, sc.fit_transform(X), y, cv=5, scoring='roc_auc')
print(f"\nBaseline (all 20 features): AUC = {scores_all.mean():.4f} ± {scores_all.std():.4f}")

## 2. Filter Methods

Filter methods rank features by a statistical score computed independently of any model. They are the fastest option and work well as a first-pass cleanup step.

**Variance threshold**: removes features with near-zero variance. A constant feature carries no information.

**Mutual Information (MI)**: measures how much knowing feature X reduces uncertainty about Y. Unlike correlation, MI captures non-linear relationships. It is the preferred filter for complex tasks.

**ANOVA F-statistic** (`f_classif`): tests whether class means differ significantly for each feature. Assumes linear relationship; fast but misses non-linear signal.

**Chi-squared test** (`chi2`): for non-negative features (counts, frequencies) and categorical targets. Tests independence; interpretable p-values.

In [ ]:
from sklearn.feature_selection import (
    VarianceThreshold, SelectKBest, mutual_info_classif, f_classif
)

# 1. Variance threshold
vt = VarianceThreshold(threshold=0.01)
vt.fit(X)
print("Features passing variance threshold:", vt.get_support().sum())

# 2. Mutual information scores
mi_scores = mutual_info_classif(X, y, random_state=42)

# 3. F-statistic scores
f_scores, f_pvals = f_classif(X, y)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
idx = np.argsort(mi_scores)[::-1]
axes[0].bar(range(20), mi_scores[idx], color='steelblue', edgecolor='k')
axes[0].set_xticks(range(20))
axes[0].set_xticklabels([feature_names[i] for i in idx], rotation=45, ha='right', fontsize=8)
axes[0].set_title('Mutual Information Score', fontsize=11)
axes[0].set_ylabel('MI Score')

idx2 = np.argsort(f_scores)[::-1]
axes[1].bar(range(20), f_scores[idx2], color='coral', edgecolor='k')
axes[1].set_xticks(range(20))
axes[1].set_xticklabels([feature_names[i] for i in idx2], rotation=45, ha='right', fontsize=8)
axes[1].set_title('ANOVA F-Statistic', fontsize=11)
axes[1].set_ylabel('F Score')

plt.suptitle('Filter Method Feature Rankings', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# SelectKBest with different k values
from sklearn.pipeline import Pipeline

print("Filter method performance (Mutual Information, SelectKBest):")
for k in [3, 5, 8, 10, 15, 20]:
    pipe = Pipeline([
        ('scale', StandardScaler()),
        ('select', SelectKBest(mutual_info_classif, k=min(k, 20))),
        ('lr', LogisticRegression(max_iter=500))
    ])
    scores = cross_val_score(pipe, X, y, cv=5, scoring='roc_auc')
    print(f"  k={k:2d}: AUC = {scores.mean():.4f} ± {scores.std():.4f}")

## 3. Wrapper Methods: Recursive Feature Elimination

**Recursive Feature Elimination (RFE)** trains a model, ranks features by importance, drops the least important feature(s), and repeats until the desired number of features remains. Because it uses the model's own ranking it naturally accounts for feature interactions — features that are useless alone may be useful in combination.

**RFECV** adds cross-validation to automatically select the optimal number of features, removing the need to pre-specify k. The cost is O(n_features × n_folds) model fits, so it is impractical with very large feature spaces — but invaluable for datasets with tens to hundreds of features.

Interview tip: RFE is most useful when you have a specific model in mind and feature interpretability matters. When you just need a parsimonious model, L1 (Lasso) is usually faster.

In [ ]:
from sklearn.feature_selection import RFE, RFECV
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# RFE with fixed k=5
lr = LogisticRegression(max_iter=500)
rfe = RFE(estimator=lr, n_features_to_select=5, step=1)
rfe.fit(X_scaled, y)
selected = [feature_names[i] for i in range(20) if rfe.support_[i]]
print("RFE selected features (k=5):", selected)
print("Feature ranking (1=selected):")
for name, rank in sorted(zip(feature_names, rfe.ranking_), key=lambda x: x[1]):
    print(f"  {name}: rank={rank}")

In [ ]:
# RFECV: auto-select optimal k via cross-validation
rfecv = RFECV(estimator=LogisticRegression(max_iter=500), step=1, cv=5,
              scoring='roc_auc', min_features_to_select=1)
rfecv.fit(X_scaled, y)

print(f"RFECV optimal number of features: {rfecv.n_features_}")
print(f"RFECV selected: {[feature_names[i] for i in range(20) if rfecv.support_[i]]}")

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(rfecv.cv_results_['mean_test_score']) + 1),
         rfecv.cv_results_['mean_test_score'], marker='o', color='steelblue')
plt.fill_between(
    range(1, len(rfecv.cv_results_['mean_test_score']) + 1),
    rfecv.cv_results_['mean_test_score'] - rfecv.cv_results_['std_test_score'],
    rfecv.cv_results_['mean_test_score'] + rfecv.cv_results_['std_test_score'],
    alpha=0.2, color='steelblue'
)
plt.axvline(rfecv.n_features_, color='red', linestyle='--', label=f'Optimal k={rfecv.n_features_}')
plt.xlabel('Number of features selected')
plt.ylabel('CV AUC')
plt.title('RFECV: Cross-Validation Score vs. Number of Features')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Embedded Methods: L1 Regularization (Lasso)

**Lasso** (L1-penalized regression) adds a penalty proportional to the absolute value of the coefficients to the loss function:

$$\mathcal{L}_{Lasso} = \text{Loss}(y, \hat{y}) + \lambda \sum_{j=1}^{p} |w_j|$$

The L1 penalty creates a **sparse solution**: many coefficients are driven to exactly zero, effectively removing those features from the model. This is in contrast to L2 (Ridge), which shrinks coefficients but rarely to exactly zero.

**Why does L1 produce zeros?** Geometrically, the L1 constraint set is a diamond (in 2D). The corners of the diamond, where coefficients are zero, are the points most likely to intersect the convex loss contours. L2's circular constraint has no corners, so zeros are improbable.

The regularization strength λ (or `C = 1/λ` in sklearn) controls sparsity: high λ → more zeros → stronger selection.

In [ ]:
from sklearn.linear_model import LogisticRegressionCV, Lasso, LassoCV
from sklearn.feature_selection import SelectFromModel

# L1 logistic regression for classification feature selection
l1_lr = LogisticRegression(penalty='l1', solver='liblinear', C=0.5, max_iter=500)
l1_lr.fit(X_scaled, y)

coef_abs = np.abs(l1_lr.coef_[0])
nonzero = (coef_abs > 0).sum()
print(f"L1 logistic regression (C=0.5): {nonzero}/20 features have non-zero coefficients")

# Plot the regularization path: vary C, observe which features survive
C_values = np.logspace(-2, 1, 30)
coef_paths = []
for C in C_values:
    m = LogisticRegression(penalty='l1', solver='liblinear', C=C, max_iter=500)
    m.fit(X_scaled, y)
    coef_paths.append(m.coef_[0])
coef_paths = np.array(coef_paths)

fig, ax = plt.subplots(figsize=(10, 5))
for i in range(20):
    ax.plot(np.log10(C_values), coef_paths[:, i], alpha=0.7, label=feature_names[i])
ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
ax.set_xlabel('log10(C)  [higher C = less regularization]')
ax.set_ylabel('Coefficient value')
ax.set_title('L1 Regularization Path — Features Enter/Exit as C Increases')
ax.legend(loc='upper left', fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
# SelectFromModel: use L1 coefficients as selection gate
sfm = SelectFromModel(LogisticRegression(penalty='l1', solver='liblinear', C=0.5, max_iter=500))
sfm.fit(X_scaled, y)
selected_l1 = [feature_names[i] for i in range(20) if sfm.get_support()[i]]
print("SelectFromModel (L1) selected:", selected_l1)

X_l1 = sfm.transform(X_scaled)
scores_l1 = cross_val_score(LogisticRegression(max_iter=500), X_l1, y, cv=5, scoring='roc_auc')
print(f"AUC with L1-selected features ({X_l1.shape[1]} features): {scores_l1.mean():.4f} ± {scores_l1.std():.4f}")

## 5. Tree-Based Feature Importance

Tree models produce feature importances as a byproduct of training. Two common types:

1. **Mean Decrease in Impurity (MDI)** — the default in sklearn. Measures how much each feature reduces impurity (Gini or entropy) averaged across all trees. Fast but biased toward high-cardinality features.

2. **Permutation importance** — shuffles each feature and measures the drop in model score. More reliable; works with any model. Computationally: O(n_features × n_repeats × n_folds).

3. **SHAP values** — provides both a global importance ranking and per-sample explanations. Gold standard for feature importance, covered in depth in the XAI series.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_scaled, y)

# MDI importance
mdi = rf.feature_importances_

# Permutation importance
perm = permutation_importance(rf, X_scaled, y, n_repeats=10, random_state=42, scoring='roc_auc')
perm_mean = perm.importances_mean

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, scores, title, color in [
    (axes[0], mdi, 'MDI (Mean Decrease Impurity)', 'steelblue'),
    (axes[1], perm_mean, 'Permutation Importance (AUC drop)', 'coral'),
]:
    idx = np.argsort(scores)[::-1]
    ax.bar(range(20), scores[idx], color=color, edgecolor='k')
    ax.set_xticks(range(20))
    ax.set_xticklabels([feature_names[i] for i in idx], rotation=45, ha='right', fontsize=8)
    ax.set_title(title, fontsize=10)

plt.suptitle('Tree-Based Feature Importance Methods', fontsize=12)
plt.tight_layout()
plt.show()
print("Note: MDI and permutation importance can disagree — permutation is more reliable for selection.")

## 6. Comparing All Methods

A good feature selection workflow: start with variance threshold (free), then apply a fast filter (MI or F), then an embedded method (L1 or tree importance), and optionally validate with RFECV. Each step is a strict subset of the previous.

In [ ]:
from sklearn.pipeline import Pipeline

results = {}

# All features
scores = cross_val_score(LogisticRegression(max_iter=500), X_scaled, y, cv=5, scoring='roc_auc')
results['All 20 features'] = (scores.mean(), scores.std(), 20)

# MI top-5
from sklearn.feature_selection import SelectKBest, mutual_info_classif
sel = SelectKBest(mutual_info_classif, k=5).fit(X_scaled, y)
scores = cross_val_score(LogisticRegression(max_iter=500), sel.transform(X_scaled), y, cv=5, scoring='roc_auc')
results['MI top-5'] = (scores.mean(), scores.std(), 5)

# RFE top-5
rfe5 = RFE(LogisticRegression(max_iter=500), n_features_to_select=5).fit(X_scaled, y)
scores = cross_val_score(LogisticRegression(max_iter=500), rfe5.transform(X_scaled), y, cv=5, scoring='roc_auc')
results['RFE top-5'] = (scores.mean(), scores.std(), 5)

# L1 selection
sfm2 = SelectFromModel(LogisticRegression(penalty='l1', solver='liblinear', C=0.5, max_iter=500)).fit(X_scaled, y)
scores = cross_val_score(LogisticRegression(max_iter=500), sfm2.transform(X_scaled), y, cv=5, scoring='roc_auc')
results[f'L1 ({sfm2.transform(X_scaled).shape[1]} feats)'] = (scores.mean(), scores.std(), sfm2.transform(X_scaled).shape[1])

# Tree importance top-5
top5_tree = np.argsort(mdi)[::-1][:5]
scores = cross_val_score(LogisticRegression(max_iter=500), X_scaled[:, top5_tree], y, cv=5, scoring='roc_auc')
results['Tree MDI top-5'] = (scores.mean(), scores.std(), 5)

print(f"{'Method':30s} | {'AUC':>8s} | {'Std':>6s} | {'n_features':>10s}")
print('-' * 62)
for method, (mean, std, nf) in results.items():
    print(f"{method:30s} | {mean:8.4f} | {std:6.4f} | {nf:10d}")

## Summary

| Method | Speed | Captures Interactions | Model-Agnostic | Best Use Case |
|--------|-------|----------------------|----------------|---------------|
| Variance Threshold | Fastest | No | Yes | Remove constants and near-constants |
| Mutual Information | Fast | Partially (pairwise) | Yes | First-pass ranking; non-linear safe |
| ANOVA F-statistic | Fast | No | Yes | Linear relationships; quick baseline |
| RFE | Slow | Yes | No (model-specific) | Small feature sets with interpretability need |
| RFECV | Slowest | Yes | No | Auto-selecting optimal k with validation |
| L1 / Lasso | Fast | Yes (within model) | No (linear) | Parsimonious linear models |
| Tree MDI importance | Fast | Yes | No (tree) | Quick ranking; biased to cardinality |
| Permutation importance | Moderate | Yes | Yes (any fitted model) | Reliable ranking; slower than MDI |